# Week 1 Lab: 2R Robot FK / IK

The goal of this session is to implement and check the relationship between joint angles and end-effector position using a simple 2R robot.

What you will implement in this notebook:
- `Planar2RRobot.fk()`
- `analytic_ik_2r()`
- `sample_workspace()`

How to use this notebook:
1. Run the cells from top to bottom.
2. Only implement the functions marked with `TODO(student)`.
3. After each implementation, use the check cells right below it to verify your result.


## What We Will Do Today

In this session, we will go through the following four steps.

1. See how the robot arm shape changes when the joint angles change.
2. Implement FK and compute the end-effector position.
3. Implement IK and find the joint angles for a target position.
4. Distinguish between reachable and unreachable targets.

The key point is not to memorize formulas, but to build intuition for how angle changes lead to position changes.


In [ ]:
# helper functions
# don't modify this block!

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
try:
    from IPython.display import HTML
except ImportError:
    def HTML(x):
        return x

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['animation.embed_limit'] = 30


def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi


def deg2rad(q):
    return np.deg2rad(q)


def rad2deg(q):
    return np.rad2deg(q)


def plot_robot(robot, q, ax=None, target_position=None, title=None, color='C0'):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    q = np.asarray(q, dtype=float)
    positions = robot.joint_positions(q)
    ee_pose = robot.fk(q)

    ax.plot(positions[:, 0], positions[:, 1], '-o', linewidth=3, markersize=8, color=color)
    ax.scatter([0], [0], s=120, marker='s', color='k')

    x, y, phi = ee_pose
    arrow_len = 0.20
    ax.arrow(x, y, arrow_len * np.cos(phi), arrow_len * np.sin(phi), head_width=0.05, length_includes_head=True, color=color)

    if target_position is not None:
        tx, ty = target_position
        ax.scatter([tx], [ty], s=120, marker='x', color='red')

    if title is not None:
        ax.set_title(title)

    reach = np.sum(robot.link_lengths) + 0.25
    ax.set_xlim(-reach, reach)
    ax.set_ylim(-reach, reach)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    return ax


## 1. 2R Robot Model

First, we define the 2R planar robot used in this notebook.

- joint angle: $q = [q_1, q_2]^T$
- link length: $l_1, l_2$
- end-effector pose: $[x, y, \phi]$

The three main equations are:

$
x = l_1\cos q_1 + l_2\cos(q_1 + q_2)
$

$
y = l_1\sin q_1 + l_2\sin(q_1 + q_2)
$

$
\phi = q_1 + q_2
$

You do not need to memorize the formulas.
It is enough to understand how the accumulated joint angles determine where the link tip moves.


### Implement: Complete the FK Function

In this cell, you will complete `Planar2RRobot.fk()`.

What you need to do is simple.

- Input: `q = [q1, q2]`
- Output: `np.array([x, y, phi])`
- The second link always points along `q1 + q2`.
- The final angle `phi` should be wrapped with `wrap_to_pi()`.

`joint_positions()` is already provided,
so here you only need to fill in the end-effector pose computation.


In [ ]:
class Planar2RRobot:
    def __init__(self, link_lengths=(1.0, 0.8), joint_limits=None):
        self.link_lengths = np.asarray(link_lengths, dtype=float)
        if joint_limits is None:
            joint_limits = np.array([
                [-np.pi, np.pi],
                [-np.pi, np.pi],
            ], dtype=float)
        self.joint_limits = np.asarray(joint_limits, dtype=float)

    def is_within_joint_limits(self, q, atol=1e-9):
        q = wrap_to_pi(np.asarray(q, dtype=float))
        lower = self.joint_limits[:, 0]
        upper = self.joint_limits[:, 1]
        return np.all(q >= lower - atol) and np.all(q <= upper + atol)

    def joint_positions(self, q):
        q = np.asarray(q, dtype=float)
        l1, l2 = self.link_lengths
        q1, q2 = q
        p0 = np.array([0.0, 0.0])
        p1 = p0 + l1 * np.array([np.cos(q1), np.sin(q1)])
        p2 = p1 + l2 * np.array([np.cos(q1 + q2), np.sin(q1 + q2)])
        return np.vstack([p0, p1, p2])

    def fk(self, q):
        q = np.asarray(q, dtype=float)
        l1, l2 = self.link_lengths
        q1, q2 = q

        # TODO(student): implement the next few lines yourself.
        # q12 = ...
        # x = ...
        # y = ...
        # phi = ...


robot = Planar2RRobot(link_lengths=(1.0, 0.8))
print('Link lengths:', robot.link_lengths)
print('Joint limits [deg]:')
print(rad2deg(robot.joint_limits))


## 2. Check the FK Implementation

If FK is implemented correctly, you should immediately see how the end-effector position changes when the joint angles change.


In [ ]:
q = deg2rad([35, 50])
pose = robot.fk(q)
positions = robot.joint_positions(q)

print('q [deg] =', rad2deg(q))
print(f'end-effector pose [x, y, phi(deg)] = [{pose[0]:.4f}, {pose[1]:.4f}, {rad2deg(pose[2]):.2f}]')
print('joint positions =')
print(positions)

fig, ax = plt.subplots(figsize=(6, 6))
plot_robot(robot, q, ax=ax, title='FK check: one robot pose')
plt.show()


## 3. Moving Demo: A Simple 2R Robot Motion Using Only FK

If the joint angles change over time, FK computes the robot pose at every instant.
The cell below shows a simple example where a sinusoidal joint motion creates an end-effector trajectory.


In [ ]:
# simple movement
sim_t = np.linspace(0, 2 * np.pi, 90)
dance_q = np.c_[deg2rad(35) + 0.9 * np.sin(sim_t), deg2rad(55) + 0.8 * np.sin(2 * sim_t + 0.6)]
dance_points = np.array([robot.joint_positions(qi) for qi in dance_q])
end_trace = dance_points[:, -1, :]

fig, ax = plt.subplots(figsize=(5, 5))
line, = ax.plot([], [], 'o-', linewidth=3, markersize=8, color='tab:blue')
trace_line, = ax.plot([], [], '--', linewidth=2, color='tab:orange', alpha=0.8)
time_text = ax.text(0.03, 0.95, '', transform=ax.transAxes)
reach = np.sum(robot.link_lengths) + 0.25
ax.set_xlim(-reach, reach)
ax.set_ylim(-reach, reach)
ax.set_aspect('equal', adjustable='box')
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('2R robot dance from FK')


def update_dance(k):
    pts = dance_points[k]
    line.set_data(pts[:, 0], pts[:, 1])
    trace_line.set_data(end_trace[: k + 1, 0], end_trace[: k + 1, 1])
    time_text.set_text(f'frame {k:02d}')
    return line, trace_line, time_text


anim = FuncAnimation(fig, update_dance, frames=len(dance_q), interval=55, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())


## 4. IK: Finding Joint Angles for a Target Point

Now let us think in the opposite direction.

So far, we computed the end-effector position when the joint angles $q$ were given.
This time, given a target position $(x, y)$, we will find the joint angles that reach that position.

The inverse kinematics process is:

1. Compute $c_2$ from the distance to the target.
2. $q_2$ usually has two branches.
3. Depending on the branch, $q_1$ also changes.


### Implement: Complete the Analytic IK Function

![alt text](figs/week1_ik.png)

A good implementation order is:

1. Compute $r^2 = x^2 + y^2$ from the target point.
2. Compute $c_2$ and check reachability.
3. Compute the two branches of $q_2$.
4. Compute $q_1$ for each branch.


In [ ]:
def analytic_ik_2r(robot, target_position, check_joint_limits=True, eps=1e-9):
    x, y = np.asarray(target_position, dtype=float)
    l1, l2 = robot.link_lengths

    # TODO(student): fill in the reachability / IK computation below.
    # 1. compute r2
    # 2. compute c2
    # 3. determine whether the target is reachable as True / False
    # 4. compute q1 and q2 for both branches
    # 5. finally return (solutions, reachable)
    # solution is represented as numpy array like: np.array([q1, q2])
    # "solutions" variable is python list that includes solution
    return solutions, reachable # python list, true/false


## 5. Same Target, Different Arm Shape

The target point below is an example where both IK branches exist.

Check whether your analytic IK works correctly, and verify visually that two different solutions can be found for the same target point.


In [ ]:
target_position = np.array([1.1, 0.7])
solutions, reachable = analytic_ik_2r(robot, target_position)

print('Target position [x, y] =', target_position)
print('Reachable =', reachable)
print('number of solutions =', len(solutions))

for i, sol in enumerate(solutions):
    fk_sol = robot.fk(sol)
    err = fk_sol[:2] - target_position
    print()
    print(f'Solution {i}')
    print('q [deg] =', rad2deg(sol))
    print('position error =', err)

fig, axes = plt.subplots(1, max(1, len(solutions)), figsize=(6 * max(1, len(solutions)), 5))
if len(solutions) == 1:
    axes = [axes]

for i, sol in enumerate(solutions):
    plot_robot(robot, sol, ax=axes[i], target_position=target_position, title=f'IK branch {i}')

plt.show()


## 6. Moving Demo: What If Two Branches Follow the Same Target Path?

Even when following the same circular target path, the arm motion can look quite different depending on the branch.
This cell gives you intuition for why `continuity of IK solutions` matters.


In [ ]:
race_center = np.array([0.95, 0.35])
race_radius = 0.18
race_angles = np.linspace(0, 2 * np.pi, 90, endpoint=False)
race_targets = race_center + race_radius * np.c_[np.cos(race_angles), np.sin(race_angles)]

race_q0, race_q1, valid_targets = [], [], []
for p in race_targets:
    sols = analytic_ik_2r(robot, p)[0]
    if len(sols) == 2:
        race_q0.append(sols[0])
        race_q1.append(sols[1])
        valid_targets.append(p)

race_q0 = np.array(race_q0)
race_q1 = np.array(race_q1)
valid_targets = np.array(valid_targets)
race_pts0 = np.array([robot.joint_positions(qi) for qi in race_q0])
race_pts1 = np.array([robot.joint_positions(qi) for qi in race_q1])

fig, ax = plt.subplots(figsize=(6, 6))
arm0, = ax.plot([], [], 'o-', linewidth=3, color='tab:blue', label='branch 0')
arm1, = ax.plot([], [], 'o-', linewidth=3, color='tab:orange', label='branch 1')
target_dot, = ax.plot([], [], 'r*', markersize=12, label='target')
ax.plot(valid_targets[:, 0], valid_targets[:, 1], 'k--', alpha=0.35, label='target path')
reach = np.sum(robot.link_lengths) + 0.25
ax.set_xlim(-reach, reach)
ax.set_ylim(-reach, reach)
ax.set_aspect('equal', adjustable='box')
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('two IK branches following the same target')
ax.legend(loc='upper right')


def update_race(k):
    arm0.set_data(race_pts0[k, :, 0], race_pts0[k, :, 1])
    arm1.set_data(race_pts1[k, :, 0], race_pts1[k, :, 1])
    target_dot.set_data([valid_targets[k, 0]], [valid_targets[k, 1]])
    return arm0, arm1, target_dot


anim = FuncAnimation(fig, update_race, frames=len(valid_targets), interval=60, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())


## 7. Workspace Sampling

Now let us visualize the approximate workspace of the robot.

The method is simple.

- Sample many joint angles.
- Compute FK for each sampled joint angle.
- Plot only the end-effector positions.

This gives a very intuitive picture of the overall reachable region.


### Implement: Complete the Workspace Sampling Function

There is not much to implement in this cell.

- random joint angle samples `qs` are already prepared
- call `robot.fk(q)` for each `q`
- collect the results into the `poses` array

This implementation is simple, but it is also a useful checkpoint to verify that the FK function you implemented earlier is working correctly.


In [ ]:
def sample_workspace(robot, num_samples=12000):
    rng = np.random.default_rng(0)
    lower = robot.joint_limits[:, 0]
    upper = robot.joint_limits[:, 1]
    # randomly sampled joint angles
    qs = rng.uniform(lower, upper, size=(num_samples, 2))

    # TODO(student): use fk to compute the end-effector pose for every q. (use qs)
    # poses = 
    return qs, poses


qs, poses = sample_workspace(robot, num_samples=12000)

plt.figure(figsize=(6, 6))
plt.scatter(poses[:, 0], poses[:, 1], s=1, alpha=0.25)
plt.axis('equal')
plt.grid(True)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Sampled workspace of the 2R robot')
plt.show()


## 8. Reachable / Unreachable at a Glance

This time, instead of looking only at the target, we will also look at the robot workspace together.

The gray points are end-effector positions obtained by randomly sampling many joint configurations.
Targets inside this point cloud are usually reachable, while targets outside it are usually unreachable.

Confirmed that even when there are no issues with the formulation or implementation, an IK solution may not exist if the target point is inherently unreachable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

reachable_target = np.array([1.1, 0.7])
unreachable_target = np.array([2.2, 0.0])

reachable_solutions, reachable_flag = analytic_ik_2r(robot, reachable_target)
solutions_bad, reachable_bad = analytic_ik_2r(robot, unreachable_target)

workspace_xy = poses[:, :2]
reach = np.sum(robot.link_lengths) + 0.5

for ax, target, title in [
    (axes[0], reachable_target, 'reachable target inside workspace'),
    (axes[1], unreachable_target, 'unreachable target outside workspace'),
]:
    ax.scatter(workspace_xy[:, 0], workspace_xy[:, 1], s=1, alpha=0.12, color='0.6', label='sampled workspace')
    ax.plot(target[0], target[1], 'r*', markersize=20, label='target')
    # ax.set_xlim(-reach, reach)
    # ax.set_ylim(-reach, reach)
    # ax.set_aspect('equal', adjustable='box')
    ax.grid(True)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title)

plot_robot(robot, reachable_solutions[0], ax=axes[0], target_position=reachable_target)
plot_robot(robot, deg2rad([0, 0]), ax=axes[1], target_position=unreachable_target)
axes[0].legend(loc='upper right')
axes[1].legend(loc='upper left')
axes[0].set_xlim(-reach, reach)
axes[0].set_ylim(-reach, reach)
axes[1].set_xlim(-reach, reach)
axes[1].set_ylim(-reach, reach)

plt.show()

print('reachable target =', reachable_target)
print('reachable target flag =', reachable_flag)
print('number of reachable solutions =', len(reachable_solutions))
print()
print('unreachable target =', unreachable_target)
print('unreachable target flag =', reachable_bad)
print('number of unreachable solutions =', len(solutions_bad))


## 9. Joint Limits and Solution Selection

On a real robot, even if multiple IK solutions exist, we do not use just any solution.

Usually we check at least the following:

- Does it satisfy the joint limits?
- Does it require too much motion from the current configuration?

This idea will remain important later when we move on to trajectory and motion planning.


In [ ]:
def choose_closest_solution(solutions, q_current):
    if len(solutions) == 0:
        return None, None

    distances = []
    for q in solutions:
        dq = wrap_to_pi(q - q_current)
        distances.append(np.linalg.norm(dq))

    best_idx = int(np.argmin(distances))
    return solutions[best_idx], distances[best_idx]


limited_robot = Planar2RRobot(
    link_lengths=(1.0, 0.8),
    joint_limits=deg2rad(np.array([
        [-180, 180],
        [0, 110],
    ])),
)

q_current = deg2rad([15, 30])
solutions_unlimited, _ = analytic_ik_2r(robot, target_position)
solutions_limited, reachable_limited = analytic_ik_2r(limited_robot, target_position)
q_best, best_dist = choose_closest_solution(solutions_unlimited, q_current)

print('Current q [deg] =', rad2deg(q_current))
print('Unlimited solutions:')
for i, sol in enumerate(solutions_unlimited):
    dist = np.linalg.norm(wrap_to_pi(sol - q_current))
    print(f'  solution {i}: q={np.round(rad2deg(sol), 2)}, distance={dist:.4f}')

print()
print('Limited robot reachable =', reachable_limited)
print('Chosen solution [deg] =', np.round(rad2deg(q_best), 2))
print('Chosen distance =', best_dist)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_robot(robot, q_current, ax=axes[0], target_position=target_position, title='current configuration')
plot_robot(robot, q_best, ax=axes[1], target_position=target_position, title='closest IK solution')
plt.show()
